# Практика 2: Реализация механизма внимания

Внимание (Attention) — ключевая часть трансформеров, позволяющая учитывать контекст всех элементов последовательности. Начинаем с создания собственного слоя внимания `MultiheadAttention` с помощью `torch.nn.Linear`. Генерируем три матрицы: `Q (запросы), K (ключи) и V (значения)`, которые получаем путём линейных преобразований входных данных. Далее выполняем матричное произведение `Q * K^T`, нормализуем результат делением на корень из размера эмбеддингов и применяем `Softmax`, чтобы получить вероятностное распределение. Умножаем на `V`, чтобы получить взвешенные значения. Реализуем `многоголовое внимание (Multi-Head Attention)`, разделяя эмбеддинги на несколько частей и применяя механизм внимания к каждой из них отдельно, после чего объединяем результаты. Проверяем работу на случайных данных, анализируем корректность формы выходного тензора. Затем интегрируем этот слой в простую нейросетевую модель и сравниваем её с обычными рекуррентными сетями.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.auto import tqdm

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

device

In [2]:
def scaled_dot_product_attention(Q, K, V, mask=None, dropout_p=0.0):
    _, _, _, D = Q.shape

    scores = torch.matmul(Q, K.transpose(-2, -1))
    scores = scores / np.sqrt(D)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    attn = torch.softmax(scores, dim=-1)

    if dropout_p > 0.0:
        attn = F.dropout(attn, p=dropout_p, training=True)

    out = torch.matmul(attn, V)
    return out, attn

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout_p: float = 0.0, bias: bool = True):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.dropout_p = dropout_p

        self.Wq = nn.Linear(d_model, d_model, bias=bias)
        self.Wk = nn.Linear(d_model, d_model, bias=bias)
        self.Wv = nn.Linear(d_model, d_model, bias=bias)
        self.Wo = nn.Linear(d_model, d_model, bias=bias)

    def _split_heads(self, x):
        B, T, _ = x.shape
        x = x.view(B, T, self.num_heads, self.head_dim)
        return x.transpose(1, 2)

    def _merge_heads(self, x):
        B, H, T, D = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(B, T, H * D)

    def forward(self, x, mask=None, need_weights=False):
        Q = self._split_heads(self.Wq(x))
        K = self._split_heads(self.Wk(x))
        V = self._split_heads(self.Wv(x))

        out, attn = scaled_dot_product_attention(Q, K, V, mask=mask, dropout_p=self.dropout_p)
        out = self._merge_heads(out)
        out = self.Wo(out)

        if need_weights:
            return out, attn
        return out

---
# Model 1

In [32]:
class TinyAttentionClassifier(nn.Module):
    def __init__(self, d_in: int, d_model: int, num_heads: int, num_classes: int):
        super().__init__()
        self.proj = nn.Linear(d_in, d_model)
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x, mask=None):
        h = self.proj(x)
        h = self.mha(h, mask=mask)
        h = self.norm(h)
        h = h.mean(dim=1)
        return self.fc(h)

# Model 2

In [33]:
class TinyGRUClassifier(nn.Module):
    def __init__(self, d_in: int, d_hidden: int, num_classes: int):
        super().__init__()
        self.gru = nn.GRU(input_size=d_in, hidden_size=d_hidden, batch_first=True)
        self.fc = nn.Linear(d_hidden, num_classes)

    def forward(self, x):
        out, h_last = self.gru(x)
        h_last = h_last.squeeze(0)
        return self.fc(h_last)

---

In [34]:
def make_synth_batch(B, T, d_in, num_classes, device):
    x = torch.randn(B, T, d_in, device=device)
    s = x.sum(dim=(1, 2))
    y = torch.bucketize(s, torch.tensor([-2.0, 2.0], device=device))
    y = torch.clamp(y, 0, num_classes - 1)
    return x, y

def train_models(att_model, rnn_model, d_in, num_classes, epochs= 200, lr= 1e-3):
    B, T = 64, 20

    opt_att = torch.optim.Adam(att_model.parameters(), lr=lr)
    opt_rnn = torch.optim.Adam(rnn_model.parameters(), lr=lr)

    loss_fn = nn.CrossEntropyLoss()

    att_model.train()
    rnn_model.train()

    for epoch in range(1,epochs + 1):
        x, y = make_synth_batch(B, T, d_in, num_classes, device)

        # Attention
        opt_att.zero_grad()
        logits_att = att_model(x)
        loss_att = loss_fn(logits_att, y)
        loss_att.backward()
        opt_att.step()

        # RNN
        opt_rnn.zero_grad()
        logits_rnn = rnn_model(x)
        loss_rnn = loss_fn(logits_rnn, y)
        loss_rnn.backward()
        opt_rnn.step()

        if epoch % 10 == 0:
            att_model.eval()
            rnn_model.eval()
            with torch.no_grad():
                acc_att = (logits_att.argmax(dim=-1) == y).float().mean().item()
                acc_rnn = (logits_rnn.argmax(dim=-1) == y).float().mean().item()
            att_model.train()
            rnn_model.train()
            print(f"step={epoch:3d} | loss_att={loss_att.item():.3f} acc_att={acc_att:.3f} | loss_rnn={loss_rnn.item():.3f} acc_rnn={acc_rnn:.3f}")

In [35]:
d_in = 16
num_classes = 3

att_model = TinyAttentionClassifier(d_in=d_in, d_model=64, num_heads=4, num_classes=num_classes).to(device)
rnn_model = TinyGRUClassifier(d_in=d_in, d_hidden=64, num_classes=num_classes).to(device)

train_models(att_model = att_model, rnn_model= rnn_model, d_in = d_in, num_classes = num_classes)

step= 10 | loss_att=0.860 acc_att=0.750 | loss_rnn=1.045 acc_rnn=0.531
step= 20 | loss_att=0.580 acc_att=0.828 | loss_rnn=0.933 acc_rnn=0.688
step= 30 | loss_att=0.473 acc_att=0.859 | loss_rnn=0.886 acc_rnn=0.578
step= 40 | loss_att=0.252 acc_att=0.938 | loss_rnn=0.809 acc_rnn=0.703
step= 50 | loss_att=0.190 acc_att=0.922 | loss_rnn=0.757 acc_rnn=0.797
step= 60 | loss_att=0.202 acc_att=0.906 | loss_rnn=0.568 acc_rnn=0.844
step= 70 | loss_att=0.117 acc_att=0.984 | loss_rnn=0.422 acc_rnn=0.875
step= 80 | loss_att=0.110 acc_att=0.969 | loss_rnn=0.444 acc_rnn=0.875
step= 90 | loss_att=0.220 acc_att=0.875 | loss_rnn=0.366 acc_rnn=0.844
step=100 | loss_att=0.146 acc_att=0.953 | loss_rnn=0.369 acc_rnn=0.891
step=110 | loss_att=0.127 acc_att=0.922 | loss_rnn=0.257 acc_rnn=0.938
step=120 | loss_att=0.089 acc_att=0.969 | loss_rnn=0.285 acc_rnn=0.891
step=130 | loss_att=0.071 acc_att=0.984 | loss_rnn=0.282 acc_rnn=0.891
step=140 | loss_att=0.081 acc_att=0.938 | loss_rnn=0.221 acc_rnn=0.922
step=1